# CPV VisDrone Training

**Phase 5 sanity:** `EPOCHS = 5`, `CONFIG = yolov8n.yaml`  
**Phase 6 full:** `EPOCHS = 50`, change `CONFIG` per model run

Inputs: `itsdreowo/cpv-visdrone5` (data), `itsdreowo/cpv-source` (code)  
No internet required.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
CONFIG    = "yolov8n.yaml"   # yolov8m.yaml | rtdetr.yaml for Phase 6
EPOCHS    = 5                # 5 = sanity, 50 = full
DATA_ROOT = "/kaggle/input/cpv-visdrone5"
DEVICE    = "cuda"

SOURCE    = "/kaggle/input/cpv-source"
WORK      = "/kaggle/working"

In [ ]:
import zipfile, shutil, subprocess, sys, os
from pathlib import Path

src_root = Path(SOURCE)
work_dir = Path(WORK)

# Show what Kaggle actually mounted so we can debug if needed
print("cpv-source contents:", sorted(p.name for p in src_root.iterdir()))

# Kaggle stores --dir-mode zip uploads as zip files; extract them to WORK
zips = sorted(src_root.glob("*.zip"))
if zips:
    for z in zips:
        print(f"Extracting {z.name} ...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(work_dir)
else:
    # Fallback: files are already directories — copy them
    for folder in ["configs", "scripts"]:
        dst = work_dir / folder
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src_root / folder, dst)

os.chdir(work_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "filterpy>=1.4.5", "-q"], check=True)
print("Setup complete. Dirs:", sorted(p.name for p in work_dir.iterdir() if p.is_dir()))

In [ ]:
# Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Train — CWD is /kaggle/working so configs/visdrone5.yaml resolves correctly
subprocess.run(
    [
        sys.executable, str(work_dir / "scripts" / "train.py"),
        "--config",    f"configs/{CONFIG}",
        "--epochs",    str(EPOCHS),
        "--device",    DEVICE,
        "--data-root", DATA_ROOT,
    ],
    check=True,
)

In [ ]:
# Show training curves
import glob
from IPython.display import Image, display

for img in sorted(glob.glob("runs/*/results.png")):
    print(img)
    display(Image(img))

In [ ]:
# Copy best weights to notebook output
for pt in sorted(Path("runs").glob("*/weights/best.pt")):
    dest = Path(WORK) / pt.parent.parent.name / "best.pt"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pt, dest)
    print("Saved:", dest)